In [13]:
import json

# 1. Định nghĩa thứ tự các điểm mốc (Bắt buộc phải cố định cho toàn bộ dataset)
# Lưu ý: Cần sửa lại nhãn "gl" trên Label Studio cho đúng với điểm thứ nhất
KEYPOINT_MAPPING = {
    "Glabella (gl) - điểm nhô giữa trán": 0,  # Điểm cao nhất trên trán (Hiện tại trong data bạn đang bị gán nhầm thành N')
    "N'": 1,
    "Pronasal (pn/Pn') - đỉnh mũi": 2,
    "Subnasale (sn) - dưới mũi": 3,
    "Labiale superius (ls) - môi trên": 4,
    "Labiale inferius (li) - môi dưới": 5,
    "B'": 6,
    "Pog'": 7
}


In [14]:
import json

# First, let's scan the annotation file to see what labels actually exist
def scan_annotations(input_file):
    """Scan and report all labels found in annotations"""
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_labels = {}
    for img_id, item in enumerate(data, start=1):
        try:
            for result in item['annotations'][0]['result']:
                if result['type'] == 'keypointlabels':
                    label = result['value']['keypointlabels'][0]
                    all_labels[label] = all_labels.get(label, 0) + 1
        except:
            pass
    
    return all_labels

# Run the scan
labels_found = scan_annotations('data/sample_train/anotation_sample_train.json')
print("📊 Labels found in your annotations:")
for label, count in sorted(labels_found.items()):
    print(f"  - '{label}': {count} occurrences")
print(f"\n❓ Labels in KEYPOINT_MAPPING but NOT in data:")
for key in KEYPOINT_MAPPING.keys():
    if key not in labels_found:
        print(f"  - '{key}' ❌ MISSING")

📊 Labels found in your annotations:
  - 'B'': 18 occurrences
  - 'Columella (cm) - trụ mũi': 1 occurrences
  - 'Glabella (gl) - điểm nhô giữa trán': 17 occurrences
  - 'Labiale inferius (li) - môi dưới': 19 occurrences
  - 'Labiale superius (ls) - môi trên': 18 occurrences
  - 'N'': 18 occurrences
  - 'Pog'': 20 occurrences
  - 'Pronasal (pn/Pn') - đỉnh mũi': 15 occurrences
  - 'Subnasale (sn) - dưới mũi': 18 occurrences

❓ Labels in KEYPOINT_MAPPING but NOT in data:


In [15]:

def convert_to_coco(label_studio_data, output_file="coco_annotations.json"):
    """
    Convert Label Studio keypoint annotations to COCO format.
    
    Requirements for COCO keypoint format:
    - keypoints: [x1, y1, v1, x2, y2, v2, ...] where v ∈ {0, 1, 2}
      * 0: not labeled
      * 1: labeled but not visible
      * 2: labeled and visible
    - bbox format: [x, y, width, height]
    - Must have same keypoint order across all images
    - VALIDATION: Each image MUST have exactly 8 keypoints (no more, no less)
    """
    EXPECTED_KEYPOINTS = len(KEYPOINT_MAPPING)  # Should be 8
    
    coco_format = {
        "info": {
            "description": "Cephalometric Facial Profile Landmarks",
            "version": "1.0"
        },
        "licenses": [],
        "categories": [{
            "id": 1,
            "name": "face",
            "keypoints": list(KEYPOINT_MAPPING.keys()),
            "skeleton": []
        }],
        "images": [],
        "annotations": []
    }

    annotation_id = 1
    skipped_images = []
    keypoint_errors = []  # Track images with wrong keypoint counts

    for img_id, item in enumerate(label_studio_data, start=1):
        try:
            # Extract image info
            filename = item.get('file_upload', f'image_{img_id}.jpg')
            if '-' in filename:
                filename = filename.split('-', 1)[1]

            # Get dimensions from first result (Label Studio format)
            orig_width = item['annotations'][0]['result'][0]['original_width']
            orig_height = item['annotations'][0]['result'][0]['original_height']

            coco_format["images"].append({
                "id": img_id,
                "file_name": filename,
                "width": int(orig_width),
                "height": int(orig_height)
            })

            # Initialize keypoints array: 8 keypoints × 3 values (x, y, visibility)
            coco_keypoints = [0.0] * (EXPECTED_KEYPOINTS * 3)
            
            xs = []
            ys = []
            found_labels = set()  # Track which keypoints we found

            # Extract keypoint landmarks
            for result in item['annotations'][0]['result']:
                if result['type'] == 'keypointlabels':
                    label = result['value']['keypointlabels'][0]
                    
                    # Check if label exists in mapping
                    if label not in KEYPOINT_MAPPING:
                        print(f"⚠️  Warning: Unknown label '{label}' in image {img_id}")
                        continue
                    
                    # Check for duplicate keypoints
                    if label in found_labels:
                        print(f"⚠️  Warning: Duplicate keypoint '{label}' in image {img_id}")
                        continue
                    
                    found_labels.add(label)
                    point_idx = KEYPOINT_MAPPING[label]
                    
                    # Convert from percentage to pixel coordinates
                    x = float(result['value']['x']) / 100.0 * orig_width
                    y = float(result['value']['y']) / 100.0 * orig_height
                    v = 2  # Visibility: 2 = labeled and visible

                    # Store in COCO format (x, y, v) for each keypoint
                    coco_keypoints[point_idx * 3] = x
                    coco_keypoints[point_idx * 3 + 1] = y
                    coco_keypoints[point_idx * 3 + 2] = v

                    xs.append(x)
                    ys.append(y)

            # ✅ VALIDATION: Check if image has exactly the expected number of keypoints
            num_keypoints = len(found_labels)
            if num_keypoints != EXPECTED_KEYPOINTS:
                error_msg = f"Image {img_id} ({filename}): has {num_keypoints} keypoints, expected {EXPECTED_KEYPOINTS}"
                missing = set(KEYPOINT_MAPPING.keys()) - found_labels
                if missing:
                    error_msg += f" | Missing: {missing}"
                keypoint_errors.append(error_msg)
                skipped_images.append(img_id)
                print(f"❌ SKIP: {error_msg}")
                continue

            # Create bounding box around keypoints with padding
            if xs and ys:
                min_x, max_x = min(xs), max(xs)
                min_y, max_y = min(ys), max(ys)
                
                box_width = max_x - min_x
                box_height = max_y - min_y
                
                # Add 20% padding on each side
                pad_x = box_width * 0.2
                pad_y = box_height * 0.2
                
                bbox_x = min_x - pad_x
                bbox_y = min_y - pad_y
                bbox_w = box_width + 2 * pad_x
                bbox_h = box_height + 2 * pad_y
                
                # Clamp bbox to image boundaries
                bbox_x = max(0, bbox_x)
                bbox_y = max(0, bbox_y)
                bbox_w = min(orig_width - bbox_x, bbox_w)
                bbox_h = min(orig_height - bbox_y, bbox_h)

                coco_format["annotations"].append({
                    "id": annotation_id,
                    "image_id": img_id,
                    "category_id": 1,
                    "keypoints": coco_keypoints,
                    "num_keypoints": len(xs),
                    "bbox": [float(bbox_x), float(bbox_y), float(bbox_w), float(bbox_h)],
                    "area": float(bbox_w * bbox_h),
                    "iscrowd": 0
                })
                annotation_id += 1
            else:
                skipped_images.append(img_id)
                print(f"⚠️  Image {img_id} ({filename}) has no keypoints - skipped")

        except Exception as e:
            print(f"❌ Error processing image {img_id}: {str(e)}")
            skipped_images.append(img_id)
            continue

    # Save to file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(coco_format, f, ensure_ascii=False, indent=4)
    
    # Print summary
    print(f"\n" + "="*80)
    print(f"✅ Conversion complete!")
    print(f"   Total images processed: {len(label_studio_data)}")
    print(f"   Total images kept:      {len(coco_format['images'])}")
    print(f"   Total annotations:      {len(coco_format['annotations'])}")
    
    if skipped_images:
        print(f"\n⚠️  Skipped {len(skipped_images)} images:")
        for error in keypoint_errors:
            print(f"     - {error}")
    
    print(f"   Output saved to: {output_file}")
    print(f"="*80)



In [16]:
# --- Cách sử dụng ---
with open('data/sample_train/anotation_sample_train.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)
convert_to_coco(raw_data)

⚠️  Warning: Unknown label 'Columella (cm) - trụ mũi' in image 2
❌ SKIP: Image 2 (TS087_20250219_Analysis_Report_img2_Lateral.jpeg): has 7 keypoints, expected 8 | Missing: {"Pronasal (pn/Pn') - đỉnh mũi"}
⚠️  Warning: Duplicate keypoint 'Pog'' in image 3
❌ SKIP: Image 3 (TS086_20250219_Analysis_Report_img2_Lateral.jpeg): has 7 keypoints, expected 8 | Missing: {"Pronasal (pn/Pn') - đỉnh mũi"}
⚠️  Warning: Duplicate keypoint 'Labiale inferius (li) - môi dưới' in image 6
❌ SKIP: Image 6 (TS081_20250219_Analysis_Report_img2_Lateral.jpeg): has 7 keypoints, expected 8 | Missing: {"N'"}
⚠️  Warning: Duplicate keypoint 'Pog'' in image 15
❌ SKIP: Image 15 (TS067_20250217_Analysis_Report_img2_Lateral.jpeg): has 7 keypoints, expected 8 | Missing: {"Pronasal (pn/Pn') - đỉnh mũi"}
⚠️  Warning: Duplicate keypoint 'N'' in image 18
❌ SKIP: Image 18 (TS060_20250214_Analysis_Report_img2_Lateral.jpeg): has 7 keypoints, expected 8 | Missing: {'Glabella (gl) - điểm nhô giữa trán'}

✅ Conversion complete!
 